In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel(
    "../data/raw/original.xls",
    header=1
)

df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [2]:
df = df.rename(columns={
    "ID": "customer_id",
    "LIMIT_BAL": "credit_limit",
    "SEX": "sex",
    "EDUCATION": "education",
    "MARRIAGE": "marriage",
    "AGE": "age",

    "PAY_0": "pay_status_sep",
    "PAY_2": "pay_status_aug",
    "PAY_3": "pay_status_jul",
    "PAY_4": "pay_status_jun",
    "PAY_5": "pay_status_may",
    "PAY_6": "pay_status_apr",

    "BILL_AMT1": "bill_sep",
    "BILL_AMT2": "bill_aug",
    "BILL_AMT3": "bill_jul",
    "BILL_AMT4": "bill_jun",
    "BILL_AMT5": "bill_may",
    "BILL_AMT6": "bill_apr",

    "PAY_AMT1": "payment_sep",
    "PAY_AMT2": "payment_aug",
    "PAY_AMT3": "payment_jul",
    "PAY_AMT4": "payment_jun",
    "PAY_AMT5": "payment_may",
    "PAY_AMT6": "payment_apr",

    "default payment next month": "default_next_month"
})

In [3]:
df.columns

Index(['customer_id', 'credit_limit', 'sex', 'education', 'marriage', 'age',
       'pay_status_sep', 'pay_status_aug', 'pay_status_jul', 'pay_status_jun',
       'pay_status_may', 'pay_status_apr', 'bill_sep', 'bill_aug', 'bill_jul',
       'bill_jun', 'bill_may', 'bill_apr', 'payment_sep', 'payment_aug',
       'payment_jul', 'payment_jun', 'payment_may', 'payment_apr',
       'default_next_month'],
      dtype='object')

In [4]:
df["sex"] = df["sex"].map({
    1: "Male",
    2: "Female"
})

In [5]:
df["sex"].value_counts()

sex
Female    18112
Male      11888
Name: count, dtype: int64

In [6]:
education_map = {
    1: "Graduate School",
    2: "University",
    3: "High School",
    4: "Other/Unknown",
    5: "Other/Unknown",
    6: "Other/Unknown",
    0: "Other/Unknown"
}

df["education"] = df["education"].map(education_map)

In [7]:
marriage_map = {
    1: "Married",
    2: "Single",
    3: "Other/Unknown",
    0: "Other/Unknown"
}

df["marriage"] = df["marriage"].map(marriage_map)

In [8]:
df["marriage"].value_counts()

marriage
Single           15964
Married          13659
Other/Unknown      377
Name: count, dtype: int64

In [9]:
age_bins = [20, 29, 39, 49, 59, float("inf")]

age_labels = [
    "21-29",
    "30-39",
    "40-49",
    "50-59",
    "60+"
]

df["age_band"] = pd.cut(
    df["age"],
    bins=age_bins,
    labels=age_labels
)

In [10]:
df["age_band"].value_counts().sort_index()

age_band
21-29     9618
30-39    11238
40-49     6464
50-59     2341
60+        339
Name: count, dtype: int64

In [11]:
print("Shape:", df.shape)

print("\nSex:")
print(df["sex"].value_counts())

print("\nEducation:")
print(df["education"].value_counts())

print("\nMarriage:")
print(df["marriage"].value_counts())

print("\nAge Bands:")
print(df["age_band"].value_counts().sort_index())

print("\nMissing values:")
print(df.isnull().sum().sum())

Shape: (30000, 26)

Sex:
sex
Female    18112
Male      11888
Name: count, dtype: int64

Education:
education
University         14030
Graduate School    10585
High School         4917
Other/Unknown        468
Name: count, dtype: int64

Marriage:
marriage
Single           15964
Married          13659
Other/Unknown      377
Name: count, dtype: int64

Age Bands:
age_band
21-29     9618
30-39    11238
40-49     6464
50-59     2341
60+        339
Name: count, dtype: int64

Missing values:
0


In [12]:
df["latest_utilization"] = (
    df["bill_sep"].clip(lower=0) / df["credit_limit"]
)

In [13]:
df["latest_utilization"].describe()

count    30000.000000
mean         0.423938
std          0.411252
min          0.000000
25%          0.022032
50%          0.313994
75%          0.829843
max          6.455300
Name: latest_utilization, dtype: float64

In [14]:
util_bins = [
    -float("inf"),
    0.30,
    0.50,
    0.75,
    1.00,
    float("inf")
]

util_labels = [
    "Low (<30%)",
    "Moderate (30-50%)",
    "Elevated (50-75%)",
    "High (75-100%)",
    "Over Limit (>100%)"
]

df["utilization_band"] = pd.cut(
    df["latest_utilization"],
    bins=util_bins,
    labels=util_labels,
    right=False
)

In [15]:
df["utilization_band"].value_counts().sort_index()

utilization_band
Low (<30%)            14792
Moderate (30-50%)      2834
Elevated (50-75%)      3579
High (75-100%)         6672
Over Limit (>100%)     2123
Name: count, dtype: int64

In [16]:
bill_cols = [
    "bill_apr",
    "bill_may",
    "bill_jun",
    "bill_jul",
    "bill_aug",
    "bill_sep"
]

df["avg_bill_6m"] = df[bill_cols].mean(axis=1)

In [17]:
payment_cols = [
    "payment_apr",
    "payment_may",
    "payment_jun",
    "payment_jul",
    "payment_aug",
    "payment_sep"
]

df["avg_payment_6m"] = df[payment_cols].mean(axis=1)

In [18]:
status_cols = [
    "pay_status_apr",
    "pay_status_may",
    "pay_status_jun",
    "pay_status_jul",
    "pay_status_aug",
    "pay_status_sep"
]
df["months_delinquent_6m"] = (
    (df[status_cols] > 0)
    .sum(axis=1)
)

In [19]:
df["max_delinquency_6m"] = (
    df[status_cols]
    .clip(lower=0)
    .max(axis=1)
)

In [20]:
df["recent_delinquency"] = (
    df["pay_status_sep"].clip(lower=0)
)

In [21]:
features = [
    "latest_utilization",
    "avg_bill_6m",
    "avg_payment_6m",
    "months_delinquent_6m",
    "max_delinquency_6m",
    "recent_delinquency"
]

df[features].describe().T

,count,mean,std,min,25%,50%,75%,max
latest_utilization,30000.0,0.423938,0.411252,0.000000,0.022032,0.313994,0.829843,6.455300
avg_bill_6m,30000.0,44976.945200,63260.721860,-56043.166667,4781.333333,21051.833333,57104.416667,877313.833333
avg_payment_6m,30000.0,5275.232094,10137.946323,0.000000,1113.291667,2397.166667,5583.916667,627344.333333
months_delinquent_6m,30000.0,0.834200,1.554303,0.000000,0.000000,0.000000,1.000000,6.000000
max_delinquency_6m,30000.0,0.682200,1.073518,0.000000,0.000000,0.000000,2.000000,8.000000
recent_delinquency,30000.0,0.356767,0.760594,0.000000,0.000000,0.000000,0.000000,8.000000


In [22]:
df["utilization_band"].value_counts().sort_index()

utilization_band
Low (<30%)            14792
Moderate (30-50%)      2834
Elevated (50-75%)      3579
High (75-100%)         6672
Over Limit (>100%)     2123
Name: count, dtype: int64

In [23]:
df["months_delinquent_6m"].value_counts().sort_index()

months_delinquent_6m
0    19931
1     4426
2     1899
3     1154
4      951
5      298
6     1341
Name: count, dtype: int64

In [24]:
df["max_delinquency_6m"].value_counts().sort_index()

max_delinquency_6m
0    19931
1     1689
2     7187
3      789
4      218
5       69
6       25
7       67
8       25
Name: count, dtype: int64

In [25]:
df.isnull().sum().sum()

np.int64(0)

In [26]:
utilization_risk = (
    df.groupby(
        "utilization_band",
        observed=True
    )["default_next_month"]
    .agg(["count", "sum", "mean"])
)

utilization_risk.columns = [
    "customers",
    "defaults",
    "default_rate"
]

utilization_risk["default_rate"] *= 100

utilization_risk

,customers,defaults,default_rate
utilization_band,,,
Low (<30%),14792,2708,18.307193
Moderate (30-50%),2834,638,22.512350
Elevated (50-75%),3579,929,25.956971
High (75-100%),6672,1723,25.824341
Over Limit (>100%),2123,638,30.051813


In [27]:
delinquency_risk = (
    df.groupby("months_delinquent_6m")
    ["default_next_month"]
    .agg(["count", "sum", "mean"])
)

delinquency_risk.columns = [
    "customers",
    "defaults",
    "default_rate"
]

delinquency_risk["default_rate"] *= 100

delinquency_risk

,customers,defaults,default_rate
months_delinquent_6m,,,
0,19931,2334,11.710401
1,4426,1320,29.823769
2,1899,736,38.757241
3,1154,587,50.866551
4,951,545,57.308097
5,298,171,57.382550
6,1341,943,70.320656


In [28]:
severity_risk = (
    df.groupby("max_delinquency_6m")
    ["default_next_month"]
    .agg(["count", "sum", "mean"])
)

severity_risk.columns = [
    "customers",
    "defaults",
    "default_rate"
]

severity_risk["default_rate"] *= 100

severity_risk

,customers,defaults,default_rate
max_delinquency_6m,,,
0,19931,2334,11.710401
1,1689,422,24.985198
2,7187,3130,43.550856
3,789,491,62.230672
4,218,140,64.220183
5,69,35,50.724638
6,25,14,56.000000
7,67,56,83.582090
8,25,14,56.000000


In [29]:
def classify_delinquency_severity(x):
    if x == 0:
        return "Current"
    elif x == 1:
        return "Mild"
    elif x == 2:
        return "Moderate"
    else:
        return "Severe"

df["delinquency_severity"] = (
    df["max_delinquency_6m"]
    .apply(classify_delinquency_severity)
)

In [30]:
severity_summary = (
    df.groupby("delinquency_severity")
      .agg(
          customers=("customer_id", "count"),
          defaults=("default_next_month", "sum"),
          default_rate=("default_next_month", "mean")
      )
)

severity_summary["default_rate"] *= 100

severity_summary

,customers,defaults,default_rate
delinquency_severity,,,
Current,19931,2334,11.710401
Mild,1689,422,24.985198
Moderate,7187,3130,43.550856
Severe,1193,750,62.866723


In [31]:
df["current_exposure"] = df["bill_sep"].clip(lower=0)

In [32]:
df["current_exposure"].describe()

count     30000.00000
mean      51246.04190
std       73608.02908
min           0.00000
25%        3558.75000
50%       22381.50000
75%       67091.00000
max      964511.00000
Name: current_exposure, dtype: float64

In [33]:
total_exposure = df["current_exposure"].sum()

total_exposure

np.int64(1537381257)

In [34]:
df["current_exposure"].mean()

np.float64(51246.0419)

In [35]:
exposure_summary = (
    df.groupby("delinquency_severity")
      .agg(
          customers=("customer_id", "count"),
          total_exposure=("current_exposure", "sum"),
          avg_exposure=("current_exposure", "mean"),
          defaults=("default_next_month", "sum"),
          default_rate=("default_next_month", "mean")
      )
)

exposure_summary["default_rate"] *= 100

exposure_summary

,customers,total_exposure,avg_exposure,defaults,default_rate
delinquency_severity,,,,,
Current,19931,1113576503,55871.582108,2334,11.710401
Mild,1689,4402433,2606.532268,422,24.985198
Moderate,7187,370902915,51607.473911,3130,43.550856
Severe,1193,48499406,40653.316010,750,62.866723


In [36]:
exposure_summary["exposure_share"] = (
    exposure_summary["total_exposure"]
    / exposure_summary["total_exposure"].sum()
    * 100
)

exposure_summary

,customers,total_exposure,avg_exposure,defaults,default_rate,exposure_share
delinquency_severity,,,,,,
Current,19931,1113576503,55871.582108,2334,11.710401,72.433334
Mild,1689,4402433,2606.532268,422,24.985198,0.286359
Moderate,7187,370902915,51607.473911,3130,43.550856,24.125630
Severe,1193,48499406,40653.316010,750,62.866723,3.154677


In [37]:
at_risk_exposure = df.loc[
    df["max_delinquency_6m"] > 0,
    "current_exposure"
].sum()

at_risk_exposure

np.int64(423804754)

In [38]:
df.loc[
    df["max_delinquency_6m"] > 0,
    "current_exposure"
]

0         3913
1         2682
8        11285
10       11073
11       12261
         ...  
29981    38671
29991     2500
29994    72557
29997     3565
29998        0
Name: current_exposure, Length: 10069, dtype: int64

In [39]:
at_risk_exposure_pct = (
    at_risk_exposure / total_exposure * 100
)

at_risk_exposure_pct

np.float64(27.56666585275015)

In [40]:
recent_delinquent_exposure = df.loc[
    df["recent_delinquency"] > 0,
    "current_exposure"
].sum()

recent_delinquent_exposure

np.int64(297721892)

In [41]:
recent_delinquent_exposure_pct = (
    recent_delinquent_exposure / total_exposure * 100
)

recent_delinquent_exposure_pct

np.float64(19.36552111874745)

In [42]:
severe_exposure = df.loc[
    df["max_delinquency_6m"] >= 3,
    "current_exposure"
].sum()

severe_exposure_pct = (
    severe_exposure / total_exposure * 100
)

severe_exposure_pct

np.float64(3.1546765500862355)

In [43]:
bill_cols = [
    "bill_apr", "bill_may", "bill_jun",
    "bill_jul", "bill_aug", "bill_sep"
]

payment_cols = [
    "payment_apr", "payment_may", "payment_jun",
    "payment_jul", "payment_aug", "payment_sep"
]

In [44]:
df["total_bill_6m"] = df[bill_cols].sum(axis=1)

df["total_payment_6m"] = df[payment_cols].sum(axis=1)

In [45]:
df["avg_positive_bill_6m"] = (
    df[bill_cols]
    .clip(lower=0)
    .mean(axis=1)
)

In [46]:
df["payment_to_bill_ratio"] = np.where(
    df["avg_positive_bill_6m"] > 0,
    df["avg_payment_6m"] / df["avg_positive_bill_6m"],
    np.nan
)

In [47]:
df["payment_to_bill_ratio"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    29034.000000
mean         0.497876
std          5.705252
min          0.000000
50%          0.095480
75%          0.627320
90%          1.000940
95%          1.188730
99%          2.735955
max        797.000000
Name: payment_to_bill_ratio, dtype: float64

In [48]:
(df["payment_to_bill_ratio"] > 1).sum()

np.int64(2938)

In [49]:
df["payment_to_bill_ratio"].isna().sum()

np.int64(966)

In [50]:
df["zero_payment_months_6m"] = (
    (df[payment_cols] == 0)
    .sum(axis=1)
)

In [51]:
df["zero_payment_months_6m"].value_counts().sort_index()

zero_payment_months_6m
0    15458
1     5473
2     3387
3     1944
4     1304
5     1002
6     1432
Name: count, dtype: int64

In [52]:
zero_payment_risk = (
    df.groupby("zero_payment_months_6m")
      .agg(
          customers=("customer_id", "count"),
          defaults=("default_next_month", "sum"),
          default_rate=("default_next_month", "mean")
      )
)

zero_payment_risk["default_rate"] *= 100

zero_payment_risk

,customers,defaults,default_rate
zero_payment_months_6m,,,
0,15458,2151,13.915125
1,5473,1586,28.978622
2,3387,1141,33.687629
3,1944,608,31.275720
4,1304,328,25.153374
5,1002,278,27.744511
6,1432,544,37.988827


In [53]:
df["recent_payment"] = df["payment_sep"]

In [54]:
df["made_recent_payment"] = np.where(
    df["recent_payment"] > 0,
    1,
    0
)

In [55]:
recent_payment_risk = (
    df.groupby("made_recent_payment")
      .agg(
          customers=("customer_id", "count"),
          defaults=("default_next_month", "sum"),
          default_rate=("default_next_month", "mean")
      )
)

recent_payment_risk["default_rate"] *= 100

recent_payment_risk

,customers,defaults,default_rate
made_recent_payment,,,
0,5249,1887,35.949705
1,24751,4749,19.187104


In [56]:
severity_payment_risk = (
    df.groupby(
        ["delinquency_severity", "made_recent_payment"]
    )
    .agg(
        customers=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean")
    )
)

severity_payment_risk["default_rate"] *= 100

severity_payment_risk

customers  defaults  default_rate
delinquency_severity made_recent_payment                                   
Current              0                         1775       335     18.873239
                     1                        18156      1999     11.010134
Mild                 0                         1093       321     29.368710
                     1                          596       101     16.946309
Moderate             0                         1720       798     46.395349
                     1                         5467      2332     42.655936
Severe               0                          661       433     65.506808
                     1                          532       317     59.586466

In [57]:
df["payment_to_bill_ratio"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    29034.000000
mean         0.497876
std          5.705252
min          0.000000
50%          0.095480
75%          0.627320
90%          1.000940
95%          1.188730
99%          2.735955
max        797.000000
Name: payment_to_bill_ratio, dtype: float64

In [58]:
df["zero_payment_months_6m"].value_counts().sort_index()

zero_payment_months_6m
0    15458
1     5473
2     3387
3     1944
4     1304
5     1002
6     1432
Name: count, dtype: int64

In [59]:
zero_payment_risk

,customers,defaults,default_rate
zero_payment_months_6m,,,
0,15458,2151,13.915125
1,5473,1586,28.978622
2,3387,1141,33.687629
3,1944,608,31.275720
4,1304,328,25.153374
5,1002,278,27.744511
6,1432,544,37.988827


In [60]:
severity_payment_risk

customers  defaults  default_rate
delinquency_severity made_recent_payment                                   
Current              0                         1775       335     18.873239
                     1                        18156      1999     11.010134
Mild                 0                         1093       321     29.368710
                     1                          596       101     16.946309
Moderate             0                         1720       798     46.395349
                     1                         5467      2332     42.655936
Severe               0                          661       433     65.506808
                     1                          532       317     59.586466

In [61]:
early_status_cols = [
    "pay_status_apr",
    "pay_status_may",
    "pay_status_jun"
]

recent_status_cols = [
    "pay_status_jul",
    "pay_status_aug",
    "pay_status_sep"
]

In [62]:
df["early_avg_delinquency"] = (
    df[early_status_cols]
    .clip(lower=0)
    .mean(axis=1)
)

df["recent_avg_delinquency"] = (
    df[recent_status_cols]
    .clip(lower=0)
    .mean(axis=1)
)

In [63]:
df["delinquency_change"] = (
    df["recent_avg_delinquency"]
    - df["early_avg_delinquency"]
)

In [64]:
def classify_trajectory(change):
    if change >= 0.5:
        return "Deteriorating"
    elif change <= -0.5:
        return "Improving"
    else:
        return "Stable"

df["delinquency_trajectory"] = (
    df["delinquency_change"]
    .apply(classify_trajectory)
)

In [65]:
df["delinquency_trajectory"].value_counts()

delinquency_trajectory
Stable           23499
Deteriorating     4228
Improving         2273
Name: count, dtype: int64

In [66]:
trajectory_risk = (
    df.groupby("delinquency_trajectory")
      .agg(
          customers=("customer_id", "count"),
          defaults=("default_next_month", "sum"),
          default_rate=("default_next_month", "mean")
      )
)

trajectory_risk["default_rate"] *= 100

trajectory_risk

,customers,defaults,default_rate
delinquency_trajectory,,,
Deteriorating,4228,2028,47.965941
Improving,2273,745,32.776067
Stable,23499,3863,16.438997


In [67]:
severity_trajectory_risk = (
    df.groupby(
        ["delinquency_severity", "delinquency_trajectory"]
    )
    .agg(
        customers=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean")
    )
)

severity_trajectory_risk["default_rate"] *= 100

severity_trajectory_risk

customers  defaults  default_rate
delinquency_severity delinquency_trajectory                                   
Current              Stable                      19931      2334     11.710401
Mild                 Deteriorating                  23         4     17.391304
                     Stable                       1666       418     25.090036
Moderate             Deteriorating                3649      1689     46.286654
                     Improving                    1930       535     27.720207
                     Stable                       1608       906     56.343284
Severe               Deteriorating                 556       335     60.251799
                     Improving                     343       210     61.224490
                     Stable                        294       205     69.727891

In [68]:
df[
    [
        "early_avg_delinquency",
        "recent_avg_delinquency",
        "delinquency_change"
    ]
].describe()

,early_avg_delinquency,recent_avg_delinquency,delinquency_change
count,30000.000000,30000.000000,30000.000000
mean,0.235611,0.326956,0.091344
std,0.652989,0.679943,0.574428
min,0.000000,0.000000,-5.333333
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.333333,0.000000
max,8.000000,7.000000,3.333333


In [69]:
df[
    [
        "pay_status_apr",
        "pay_status_may",
        "pay_status_jun",
        "pay_status_jul",
        "pay_status_aug",
        "pay_status_sep",
        "delinquency_trajectory"
    ]
].sample(10, random_state=42)

,pay_status_apr,pay_status_may,pay_status_jun,pay_status_jul,pay_status_aug,pay_status_sep,delinquency_trajectory
2308,0,0,0,0,0,0,Stable
22404,0,0,0,0,0,0,Stable
23397,0,0,0,0,0,0,Stable
25058,-1,0,0,0,0,0,Stable
2664,2,0,0,0,0,0,Improving
8511,0,0,0,-1,-1,-1,Stable
5148,-2,0,0,0,2,1,Deteriorating
7790,-2,-2,-2,-2,-2,1,Stable
11311,0,-1,-1,-1,-1,-1,Stable
19043,-2,-2,-2,-2,0,0,Stable


In [70]:
trajectory_risk

,customers,defaults,default_rate
delinquency_trajectory,,,
Deteriorating,4228,2028,47.965941
Improving,2273,745,32.776067
Stable,23499,3863,16.438997


In [71]:
severity_trajectory_risk

customers  defaults  default_rate
delinquency_severity delinquency_trajectory                                   
Current              Stable                      19931      2334     11.710401
Mild                 Deteriorating                  23         4     17.391304
                     Stable                       1666       418     25.090036
Moderate             Deteriorating                3649      1689     46.286654
                     Improving                    1930       535     27.720207
                     Stable                       1608       906     56.343284
Severe               Deteriorating                 556       335     60.251799
                     Improving                     343       210     61.224490
                     Stable                        294       205     69.727891

In [72]:
severity_score_map = {
    "Current": 0,
    "Mild": 1,
    "Moderate": 2,
    "Severe": 3
}

df["severity_score"] = (
    df["delinquency_severity"]
    .map(severity_score_map)
)

In [73]:
def score_frequency(months):
    if months == 0:
        return 0
    elif months == 1:
        return 1
    elif months <= 3:
        return 2
    else:
        return 3

df["frequency_score"] = (
    df["months_delinquent_6m"]
    .apply(score_frequency)
)

In [74]:
df["recent_delinquency"]

0        2
1        0
2        0
3        0
4        0
        ..
29995    0
29996    0
29997    4
29998    1
29999    0
Name: recent_delinquency, Length: 30000, dtype: int64

In [75]:
def score_recent_delinquency(x):
    if x == 0:
        return 0
    elif x == 1:
        return 1
    else:
        return 2

df["recent_delinquency_score"] = (
    df["recent_delinquency"]
    .apply(score_recent_delinquency)
)

In [76]:
df["payment_behavior_score"] = np.where(
    df["made_recent_payment"] == 0,
    1,
    0
)

In [77]:
df["utilization_score"] = np.where(
    df["latest_utilization"] >= 0.75,
    1,
    0
)

In [78]:
df["trajectory_score"] = np.where(
    df["delinquency_trajectory"] == "Deteriorating",
    1,
    0
)

In [79]:
df["behavioral_risk_score"] = (
    df["severity_score"]
    + df["frequency_score"]
    + df["recent_delinquency_score"]
    + df["payment_behavior_score"]
    + df["utilization_score"]
    + df["trajectory_score"]
)

In [80]:
risk_score_validation = (
    df.groupby("behavioral_risk_score")
      .agg(
          customers=("customer_id", "count"),
          defaults=("default_next_month", "sum"),
          default_rate=("default_next_month", "mean"),
          total_exposure=("current_exposure", "sum")
      )
)

risk_score_validation["default_rate"] *= 100

risk_score_validation

,customers,defaults,default_rate,total_exposure
behavioral_risk_score,,,,
0,12659,1205,9.518919,473498941
1,7200,1119,15.541667,633278223
2,72,10,13.888889,6799339
3,1350,263,19.481481,28071161
4,2492,653,26.203852,59738519
5,868,279,32.142857,34227668
6,1125,534,47.466667,39502222
7,1765,1011,57.280453,107984238
8,1334,803,60.194903,83369708


In [81]:
df.groupby("behavioral_risk_score")[
    "default_next_month"
].mean() * 100

behavioral_risk_score
0      9.518919
1     15.541667
2     13.888889
3     19.481481
4     26.203852
5     32.142857
6     47.466667
7     57.280453
8     60.194903
9     69.419355
10    62.400000
11    59.090909
Name: default_next_month, dtype: float64

In [82]:
df["behavioral_risk_score"].describe()

count    30000.000000
mean         2.205467
std          2.879478
min          0.000000
25%          0.000000
50%          1.000000
75%          4.000000
max         11.000000
Name: behavioral_risk_score, dtype: float64

In [83]:
df["behavioral_risk_score"].value_counts().sort_index()

behavioral_risk_score
0     12659
1      7200
2        72
3      1350
4      2492
5       868
6      1125
7      1765
8      1334
9       775
10      250
11      110
Name: count, dtype: int64

In [84]:
risk_score_validation

,customers,defaults,default_rate,total_exposure
behavioral_risk_score,,,,
0,12659,1205,9.518919,473498941
1,7200,1119,15.541667,633278223
2,72,10,13.888889,6799339
3,1350,263,19.481481,28071161
4,2492,653,26.203852,59738519
5,868,279,32.142857,34227668
6,1125,534,47.466667,39502222
7,1765,1011,57.280453,107984238
8,1334,803,60.194903,83369708


In [85]:
def classify_risk(score):
    if score <= 3:
        return "Low"
    elif score <= 5:
        return "Medium"
    else:
        return "High"

df["risk_segment"] = (
    df["behavioral_risk_score"]
    .apply(classify_risk)
)

In [86]:
risk_segment_summary = (
    df.groupby("risk_segment")
      .agg(
          customers=("customer_id", "count"),
          defaults=("default_next_month", "sum"),
          default_rate=("default_next_month", "mean"),
          total_exposure=("current_exposure", "sum"),
          avg_exposure=("current_exposure", "mean")
      )
)

risk_segment_summary["default_rate"] *= 100

risk_segment_summary

,customers,defaults,default_rate,total_exposure,avg_exposure
risk_segment,,,,,
High,5359,3107,57.977235,301767406,56310.394850
Low,21281,2597,12.203374,1141647664,53646.335417
Medium,3360,932,27.738095,93966187,27966.127083


In [87]:
risk_segment_summary["exposure_share"] = (
    risk_segment_summary["total_exposure"]
    / risk_segment_summary["total_exposure"].sum()
    * 100
)

risk_segment_summary

,customers,defaults,default_rate,total_exposure,avg_exposure,exposure_share
risk_segment,,,,,,
High,5359,3107,57.977235,301767406,56310.394850,19.628664
Low,21281,2597,12.203374,1141647664,53646.335417,74.259242
Medium,3360,932,27.738095,93966187,27966.127083,6.112094


In [88]:
df["current_exposure"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count     30000.00000
mean      51246.04190
std       73608.02908
min           0.00000
25%        3558.75000
50%       22381.50000
75%       67091.00000
90%      142133.70000
95%      201203.05000
99%      350110.68000
max      964511.00000
Name: current_exposure, dtype: float64

In [89]:
df.groupby("risk_segment")["current_exposure"].describe(
    percentiles=[0.50, 0.75, 0.90]
)

,count,mean,std,min,50%,75%,90%,max
risk_segment,,,,,,,,
High,5359.0,56310.394850,71701.299325,0.0,30335.0,72107.5,138853.2,613860.0
Low,21281.0,53646.335417,76306.402897,0.0,23070.0,70844.0,150426.0,964511.0
Medium,3360.0,27966.127083,51778.651489,0.0,5003.5,32236.5,86740.0,474934.0


In [90]:
low_threshold = df["current_exposure"].quantile(0.75)
high_threshold = df["current_exposure"].quantile(0.90)

print("75th percentile:", low_threshold)
print("90th percentile:", high_threshold)

75th percentile: 67091.0
90th percentile: 142133.70000000004


In [91]:
def classify_exposure(exposure):
    if exposure <= low_threshold:
        return "Low"
    elif exposure <= high_threshold:
        return "Medium"
    else:
        return "High"

df["exposure_band"] = (
    df["current_exposure"]
    .apply(classify_exposure)
)

In [92]:
df["exposure_band"].value_counts()

exposure_band
Low       22500
Medium     4500
High       3000
Name: count, dtype: int64

In [93]:
risk_exposure_matrix = (
    df.groupby(
        ["risk_segment", "exposure_band"]
    )
    .agg(
        customers=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean"),
        total_exposure=("current_exposure", "sum")
    )
)

risk_exposure_matrix["default_rate"] *= 100

risk_exposure_matrix

customers  defaults  default_rate  total_exposure
risk_segment exposure_band                                                   
High         High                 524       308     58.778626       120438244
             Low                 3908      2236     57.215967        92308861
             Medium               927       563     60.733549        89020301
Low          High                2341       254     10.850064       538833933
             Low                15674      2059     13.136404       282022680
             Medium              3266       284      8.695652       320791051
Medium       High                 135        32     23.703704        29406992
             Low                 2918       838     28.718300        33818064
             Medium               307        62     20.195440        30741131

In [94]:
def assign_collection_priority(row):

    risk = row["risk_segment"]
    exposure = row["exposure_band"]

    if risk == "High" and exposure == "High":
        return "Critical"

    elif risk == "High" and exposure == "Medium":
        return "High Priority"

    elif risk == "High" and exposure == "Low":
        return "Priority"

    elif risk == "Medium" and exposure in ["Medium", "High"]:
        return "Priority"

    elif risk == "Medium" and exposure == "Low":
        return "Monitor"

    elif risk == "Low" and exposure == "High":
        return "Monitor"

    else:
        return "Routine"

In [95]:
df["collection_priority"] = df.apply(
    assign_collection_priority,
    axis=1
)

In [96]:
collection_summary = (
    df.groupby("collection_priority")
      .agg(
          customers=("customer_id", "count"),
          defaults=("default_next_month", "sum"),
          default_rate=("default_next_month", "mean"),
          total_exposure=("current_exposure", "sum"),
          avg_exposure=("current_exposure", "mean")
      )
)

collection_summary["default_rate"] *= 100

collection_summary

,customers,defaults,default_rate,total_exposure,avg_exposure
collection_priority,,,,,
Critical,524,308,58.778626,120438244,229843.977099
High Priority,927,563,60.733549,89020301,96030.529666
Monitor,5259,1092,20.764404,572651997,108889.902453
Priority,4350,2330,53.563218,152456984,35047.582529
Routine,18940,2343,12.370644,602813731,31827.546515


In [97]:
collection_summary["exposure_share"] = (
    collection_summary["total_exposure"]
    / df["current_exposure"].sum()
    * 100
)

collection_summary

,customers,defaults,default_rate,total_exposure,avg_exposure,exposure_share
collection_priority,,,,,,
Critical,524,308,58.778626,120438244,229843.977099,7.833987
High Priority,927,563,60.733549,89020301,96030.529666,5.790385
Monitor,5259,1092,20.764404,572651997,108889.902453,37.248535
Priority,4350,2330,53.563218,152456984,35047.582529,9.916667
Routine,18940,2343,12.370644,602813731,31827.546515,39.210425


In [98]:
collection_summary["customer_share"] = (
    collection_summary["customers"]
    / len(df)
    * 100
)

collection_summary

,customers,defaults,default_rate,total_exposure,avg_exposure,exposure_share,customer_share
collection_priority,,,,,,,
Critical,524,308,58.778626,120438244,229843.977099,7.833987,1.746667
High Priority,927,563,60.733549,89020301,96030.529666,5.790385,3.090000
Monitor,5259,1092,20.764404,572651997,108889.902453,37.248535,17.530000
Priority,4350,2330,53.563218,152456984,35047.582529,9.916667,14.500000
Routine,18940,2343,12.370644,602813731,31827.546515,39.210425,63.133333


In [99]:
df.columns.tolist()

['customer_id',
 'credit_limit',
 'sex',
 'education',
 'marriage',
 'age',
 'pay_status_sep',
 'pay_status_aug',
 'pay_status_jul',
 'pay_status_jun',
 'pay_status_may',
 'pay_status_apr',
 'bill_sep',
 'bill_aug',
 'bill_jul',
 'bill_jun',
 'bill_may',
 'bill_apr',
 'payment_sep',
 'payment_aug',
 'payment_jul',
 'payment_jun',
 'payment_may',
 'payment_apr',
 'default_next_month',
 'age_band',
 'latest_utilization',
 'utilization_band',
 'avg_bill_6m',
 'avg_payment_6m',
 'months_delinquent_6m',
 'max_delinquency_6m',
 'recent_delinquency',
 'delinquency_severity',
 'current_exposure',
 'total_bill_6m',
 'total_payment_6m',
 'avg_positive_bill_6m',
 'payment_to_bill_ratio',
 'zero_payment_months_6m',
 'recent_payment',
 'made_recent_payment',
 'early_avg_delinquency',
 'recent_avg_delinquency',
 'delinquency_change',
 'delinquency_trajectory',
 'severity_score',
 'frequency_score',
 'recent_delinquency_score',
 'payment_behavior_score',
 'utilization_score',
 'trajectory_score',
 'b

In [100]:
customer_cols = [
    "customer_id",
    "credit_limit",
    "sex",
    "education",
    "marriage",
    "age",
    "age_band"
]

financial_cols = [
    "bill_sep",
    "bill_aug",
    "bill_jul",
    "bill_jun",
    "bill_may",
    "bill_apr",
    "payment_sep",
    "payment_aug",
    "payment_jul",
    "payment_jun",
    "payment_may",
    "payment_apr",
    "current_exposure"
]

behavior_cols = [
    "latest_utilization",
    "utilization_band",
    "avg_bill_6m",
    "avg_payment_6m",
    "months_delinquent_6m",
    "max_delinquency_6m",
    "recent_delinquency",
    "delinquency_severity",
    "zero_payment_months_6m",
    "made_recent_payment",
    "delinquency_trajectory"
]

risk_cols = [
    "behavioral_risk_score",
    "risk_segment",
    "exposure_band",
    "collection_priority",
    "default_next_month"
]

final_cols = (
    customer_cols
    + financial_cols
    + behavior_cols
    + risk_cols
)

analytics_df = df[final_cols].copy()

In [101]:
analytics_df.shape

(30000, 36)

In [103]:
analytics_df.head()

,customer_id,credit_limit,sex,education,marriage,age,age_band,bill_sep,bill_aug,bill_jul,...,delinquency_severity,zero_payment_months_6m,made_recent_payment,delinquency_trajectory,behavioral_risk_score,risk_segment,exposure_band,collection_priority,default_next_month,recent_delinquency_status
0,1,20000,Female,University,Married,24,21-29,3913,3102,689,...,Moderate,5,0,Deteriorating,8,High,Low,Priority,1,2 Months Delinquent
1,2,120000,Female,University,Single,26,21-29,2682,1725,2682,...,Moderate,2,0,Stable,5,Medium,Low,Monitor,1,Current
2,3,90000,Female,University,Single,34,30-39,29239,14027,13559,...,Current,0,1,Stable,0,Low,Low,Routine,0,Current
3,4,50000,Female,University,Married,37,30-39,46990,48233,49291,...,Current,0,1,Stable,1,Low,Low,Routine,0,Current
4,5,50000,Male,University,Married,57,50-59,8617,5670,35835,...,Current,0,1,Stable,0,Low,Low,Routine,0,Current


In [102]:
def recent_status_label(x):
    if x <= 0:
        return "Current"
    elif x == 1:
        return "1 Month Delinquent"
    elif x == 2:
        return "2 Months Delinquent"
    else:
        return "3+ Months Delinquent"

analytics_df["recent_delinquency_status"] = (
    analytics_df["recent_delinquency"]
    .apply(recent_status_label)
)

In [104]:
analytics_df[
    ["recent_delinquency", "recent_delinquency_status"]
].value_counts().sort_index()

recent_delinquency  recent_delinquency_status
0                   Current                      23182
1                   1 Month Delinquent            3688
2                   2 Months Delinquent           2667
3                   3+ Months Delinquent           322
4                   3+ Months Delinquent            76
5                   3+ Months Delinquent            26
6                   3+ Months Delinquent            11
7                   3+ Months Delinquent             9
8                   3+ Months Delinquent            19
Name: count, dtype: int64

In [105]:
print("Shape:", analytics_df.shape)

print("\nDuplicate customers:")
print(analytics_df["customer_id"].duplicated().sum())

print("\nTotal missing values:")
print(analytics_df.isnull().sum().sum())

print("\nRisk segments:")
print(analytics_df["risk_segment"].value_counts())

print("\nCollection priorities:")
print(analytics_df["collection_priority"].value_counts())

Shape: (30000, 37)

Duplicate customers:
0

Total missing values:
0

Risk segments:
risk_segment
Low       21281
High       5359
Medium     3360
Name: count, dtype: int64

Collection priorities:
collection_priority
Routine          18940
Monitor           5259
Priority          4350
High Priority      927
Critical           524
Name: count, dtype: int64


In [106]:
print("CUSTOMER CHECK")
print("Original:", len(df))
print("Final:", len(analytics_df))

print("\nEXPOSURE CHECK")
print("Original:", df["current_exposure"].sum())
print("Final:", analytics_df["current_exposure"].sum())

print("\nDEFAULT CHECK")
print("Original:", df["default_next_month"].sum())
print("Final:", analytics_df["default_next_month"].sum())

CUSTOMER CHECK
Original: 30000
Final: 30000

EXPOSURE CHECK
Original: 1537381257
Final: 1537381257

DEFAULT CHECK
Original: 6636
Final: 6636


In [107]:
from pathlib import Path

Path("../data/processed").mkdir(
    parents=True,
    exist_ok=True
)

In [108]:
analytics_df.to_csv(
    "../data/processed/credit_risk_analytics.csv",
    index=False
)

In [109]:
from pathlib import Path

file_path = Path(
    "../data/processed/credit_risk_analytics.csv"
)

print("Exists:", file_path.exists())
print("Size:", file_path.stat().st_size)

Exists: True
Size: 6793569
